In [3]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

S = 1237  # seed set to 1237
torch.manual_seed(S)

X, y = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)  # Bug 2: scaling before split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=S, stratify=y
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(32, 10),
    nn.Softmax(dim=1),  # Bug 3: extra Softmax before CrossEntropyLoss
)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    opt.step()

# Bug 4 & 5: evaluating on train set without model.eval()
pred = model(X_train).argmax(dim=1)
acc = (pred == y_train).float().mean().item()
print("test accuracy:", round(acc, 3))

test accuracy: 0.705


In [4]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

S = 1237
torch.manual_seed(S)

X, y = load_digits(return_X_y=True)

# Fix 2: split first, fit scaler ONLY on X_train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=S, stratify=y
)

scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(32, 10),
    nn.Softmax(dim=1),  # still present for next step
)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    opt.step()

# Still original evaluation lines for now
pred = model(X_train).argmax(dim=1)
acc = (pred == y_train).float().mean().item()
print("test accuracy:", round(acc, 3))

test accuracy: 0.702


In [5]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

S = 1237
torch.manual_seed(S)

X, y = load_digits(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=S, stratify=y
)

scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

# Fix 3: removed redundant Softmax(dim=1) before CrossEntropyLoss
model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(32, 10),
)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    opt.step()

# Still train-set eval without model.eval() for next steps
pred = model(X_train).argmax(dim=1)
acc = (pred == y_train).float().mean().item()
print("test accuracy:", round(acc, 3))

test accuracy: 0.898


In [6]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

S = 1237
torch.manual_seed(S)

X, y = load_digits(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=S, stratify=y
)

scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(32, 10),
)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    opt.step()

# Fix 4: evaluate on X_test and y_test instead of X_train and y_train
pred = model(X_test).argmax(dim=1)
acc = (pred == y_test).float().mean().item()
print("test accuracy:", round(acc, 3))

test accuracy: 0.842


In [7]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

S = 1237
torch.manual_seed(S)

X, y = load_digits(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=S, stratify=y
)

scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(32, 10),
)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    model.train()
    opt.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    opt.step()

# Fix 4: set eval mode and disable gradients during evaluation
model.eval()
with torch.no_grad():
    pred = model(X_test).argmax(dim=1)
    acc = (pred == y_test).float().mean().item()

print("test accuracy:", round(acc, 3))

test accuracy: 0.889


### Part E: Bug Hunt (`S = 1237`)

* **Initial Broken Script Accuracy:** `[fill printed value, ~0.70]`
* **Final Fixed Script Accuracy (Test Split):** `[fill true test accuracy, ~0.90+]`

| # | Buggy Line / Area | Why It Is Wrong | Concept Behind It | Accuracy Before → After |
| :- | :--- | :--- | :--- | :---: |
| 1 | `X = StandardScaler().fit_transform(X)` before split | Fits scaler on the full dataset including test samples before splitting | Data leakage / train-test separation | `[val]` → `[val]` |
| 2 | `nn.Softmax(dim=1)` at end of `nn.Sequential` with `nn.CrossEntropyLoss()` | `CrossEntropyLoss` expects raw logits; double-applying softmax distorts gradients/loss | Logits vs. probabilities & loss formulation | `[val]` → `[val]` |
| 3 | `opt = torch.optim.SGD(model.parameters(), lr=0.1)` without `model.train()` / `model.eval()` | `Dropout(0.5)` stays active during evaluation/prediction, dropping half the activations | Train vs. eval mode & dropout inference | `[val]` → `[val]` |
| 4 | `pred = model(X_train).argmax(dim=1)` and `y_train` used to compute accuracy | Evaluates training accuracy on `X_train`/`y_train` instead of unseen `X_test`/`y_test` | Training vs. test evaluation split | `[val]` → `[val]` |
| 5 | `[fill 5th bug line from your script]` | `[fill why it is wrong]` | `[fill ML/DL concept]` | `[val]` → `[val]` |

* **Most Dangerous Bug:** Evaluating on `X_train`/`y_train` (or leaking test data via `StandardScaler` before split) makes the reported accuracy look artificially high while hiding poor generalization on unseen data—dangerous in real work because a model that fails in production can be deployed with false confidence.